# 03 — Model training, evaluation, and comparison

Train all four classifiers (Naive Bayes, Logistic Regression, Linear SVM, Random Forest), score them on the held-out test split, and pick a winner.

Each algorithm gets a **markdown explanation cell BEFORE its training cell** that covers what it does, where it's used in the wider world, and why it earns a spot in this comparison. The metric definitions and trade-offs sit just before the evaluation section. After all models are scored, a final cell discusses the metric hierarchy specific to disaster-tweet classification and the rationale for the winner.

## 0. Setup

Import everything, load the cleaned data, build the TF-IDF features, and split using CrisisBench's official train/test split (no re-shuffling — the original split is the recognised benchmark).

In [1]:
import os, sys, time
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocess import preprocess_dataframe
from src.features import build_tfidf, transform_tfidf
from src.train import MODEL_CONFIGS, train_model, save_model
from src.evaluate import (
    evaluate_model, plot_confusion_matrix, plot_classification_report,
    plot_per_class_f1, compare_models, get_misclassified,
)
from src.utils import (
    PROCESSED_DATA_DIR, FIGURES_DIR, MODELS_DIR,
    set_plot_style, ensure_dir, log_step,
)

set_plot_style()
for d in (FIGURES_DIR, MODELS_DIR):
    ensure_dir(d)

df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'crisisbench_en.csv'))
df = preprocess_dataframe(df)

train_df = df[df['split'] == 'train'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)
y_train, y_test = train_df['class_label'].values, test_df['class_label'].values
print(f'train={len(train_df):,}  test={len(test_df):,}')

  preprocess_dataframe: dropping 18 rows with empty cleaned text
train=92,743  test=26,292


In [2]:
# Build TF-IDF on the training split, then transform the test split using
# the SAME fitted vectoriser. This is critical — the test data must live in
# the same feature space the model was trained in.
vec, X_train = build_tfidf(train_df['text_clean'])
X_test = transform_tfidf(vec, test_df['text_clean'])
print(f'X_train shape : {X_train.shape}   ({X_train.shape[0]:,} tweets × {X_train.shape[1]:,} features)')
print(f'X_test shape  : {X_test.shape}')
print(f'Sparsity      : {1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1]):.4%}')

# We'll evaluate against the union of labels seen in either split, so that
# rare classes still appear in the per-class plots.
labels = sorted(set(y_train) | set(y_test))
print(f'Class labels ({len(labels)}): {labels}')

X_train shape : (92743, 10000)   (92,743 tweets × 10,000 features)
X_test shape  : (26292, 10000)
Sparsity      : 99.9141%
Class labels (16): ['affected_individual', 'caution_and_advice', 'disease_related', 'displaced_and_evacuations', 'donation_and_volunteering', 'infrastructure_and_utilities_damage', 'injured_or_dead_people', 'missing_and_found_people', 'not_humanitarian', 'other_relevant_information', 'personal_update', 'physical_landslide', 'requests_or_needs', 'response_efforts', 'sympathy_and_support', 'terrorism_related']


## Model 1 — Naive Bayes (MultinomialNB)

**What it does, in plain English.** Bayes' rule says `P(class | tweet) ∝ P(class) × P(tweet | class)`. To compute `P(tweet | class)` exactly we'd need the joint distribution of every word combination — astronomically expensive. The *naive* assumption is that words in a tweet are conditionally INDEPENDENT given the class, so we can multiply per-word probabilities instead. That's wildly false in real language ("storm" and "surge" are obviously correlated), but it produces a model that is fast, simple, and surprisingly competitive on text.

**Worked example for** `"Bridge collapsed in earthquake, people trapped"`. Suppose the model has counted, in training data:
- `P(infrastructure_damage)` = 4% (class prior)
- `P("bridge" | infra)`     = 0.012
- `P("collapsed" | infra)`  = 0.018
- `P("earthquake" | infra)` = 0.025
- `P("people" | infra)`     = 0.04
- `P("trapped" | infra)`    = 0.009

Score for `infrastructure_damage` ∝ 0.04 × 0.012 × 0.018 × 0.025 × 0.04 × 0.009 ≈ 7.8 × 10⁻¹². Tiny in absolute terms, but compared with the same product for `donation_and_volunteering` (where most of those word probabilities are far smaller), the ratio is huge — and that ratio is what `argmax` picks.

**Why "Multinomial".** `MultinomialNB` is the variant tailored to count / frequency features. Bernoulli NB models word *presence* (binary), Gaussian NB models continuous-valued features. TF-IDF isn't strictly a count, but it's a non-negative real-valued frequency proxy that `MultinomialNB` handles fine because the implementation only needs feature values to be ≥ 0.

**Where it shows up in the wild.** Spam filtering (the original killer app), email triage, news topic classification, sentiment analysis, simple medical triage. Anywhere you want a fast text classifier with no GPU.

**Why it's in our comparison.** It's the **baseline floor**. Anything we deploy must beat NB; if it doesn't, the added complexity isn't earning its keep. It's also the only model in the four that has *no* class-weight knob, so it gives us a clean reading of what unweighted training looks like.

In [3]:
nb_model, nb_time = train_model('Naive Bayes', MODEL_CONFIGS['Naive Bayes'], X_train, y_train)

[15:51:14]   training Naive Bayes …
[15:51:14]     Naive Bayes done in 0.2s


## Model 2 — Logistic Regression

**What it does, in plain English.** For each (word, class) pair, the model learns one weight. To classify a tweet, it sums the weights of every word in the tweet (per class) and runs the resulting score-per-class through a softmax to get probabilities.

**Worked example for** `"Bridge collapsed in earthquake, people trapped"`. Suppose the model has learned, for class `infrastructure_damage`:

| word | weight |
|---|---|
| bridge | +1.4 |
| collapsed | +2.1 |
| earthquake | +0.6 |
| people | -0.1 |
| trapped | +0.9 |

Sum = +4.9. Combined with the bias term that's a very large logit, which softmax turns into a probability close to 1 for that class. The same words yield much smaller sums for `donation_and_volunteering`, so it loses.

**Where it shows up in the wild.** Spam filtering, click-through-rate prediction, churn modelling, credit scoring, A/B test analysis, medical risk scoring. The workhorse of applied ML — fast, interpretable, scales to billions of features.

**Why it's in our comparison.** Top performer on TF-IDF in most published text-classification benchmarks (including the original CrisisBench paper), and *interpretable*: per-class word weights mean we can inspect why any prediction came out the way it did, which matters in disaster-response settings where humans review model output.

In [4]:
lr_model, lr_time = train_model('Logistic Regression', MODEL_CONFIGS['Logistic Regression'], X_train, y_train)

[15:51:14]   training Logistic Regression …


[15:51:19]     Logistic Regression done in 5.3s


## Model 3 — Linear SVM (LinearSVC)

**What it does, in plain English.** A support-vector machine looks for the boundary that maximally separates the classes — the line (or hyperplane in higher dimensions) with the *largest possible margin* between the closest points of opposing classes. The closest points are the "support vectors"; everything farther away doesn't influence the boundary.

**Worked example.** Imagine plotting tweets in 2D (we actually have 10,000 dimensions, but two are easier to picture). `infrastructure_damage` tweets cluster on one side of the plane, `donation_and_volunteering` tweets on the other. SVM draws a line such that the closest tweet on each side is as far from the line as possible. A new tweet is classified by which side of the line it lands on. The further from the line, the more confident.

**Why "Linear".** A general SVM can use a non-linear *kernel* to bend the boundary. For sparse high-dimensional text, the linear kernel is empirically best — text already lives in a very high-dimensional space, and bending the boundary further usually overfits.

**How it differs from Logistic Regression.** Both produce linear decision boundaries, but they minimise different losses. LR minimises log-loss (well-calibrated probabilities), SVM minimises hinge loss (margin maximisation). LR cares about getting probabilities right; SVM only cares about getting points on the correct side of the margin.

**Where it shows up in the wild.** Bioinformatics (gene classification), image categorisation (pre-deep-learning era), text classification, handwritten-digit recognition.

**Why it's in our comparison.** Replicates the methodology used in the published CrisisBench paper, and provides a *different optimisation philosophy* than LR. They often score very similarly on text but their failure modes differ — a useful diagnostic.

In [5]:
svm_model, svm_time = train_model('Linear SVM', MODEL_CONFIGS['Linear SVM'], X_train, y_train)

[15:51:19]   training Linear SVM …


[15:51:25]     Linear SVM done in 5.7s


## Model 4 — Random Forest

**What a decision tree does.** A decision tree asks a sequence of yes/no questions about the features. For example: *"does the tweet contain 'help'?"* → yes → *"does it contain 'donate'?"* → no → *"does it contain 'trapped'?"* → yes → predict `missing_and_found_people`. Each internal node is a feature threshold; each leaf is a class prediction.

**What a random forest does.** A random forest trains many decision trees on different bootstrap samples of the data and on random subsets of features, then takes a majority vote over their predictions. Individual trees can wildly overfit, but the randomness across trees decorrelates their errors, so the majority vote is much more stable. This trick — combining many noisy, diverse, weak learners into one strong learner — is called **ensembling**.

**Worked example.** For `"Bridge collapsed in earthquake, people trapped"`, tree 1 might split on "trapped" → "earthquake" → predict `infrastructure_damage`. Tree 2 might split on "collapsed" → "bridge" → also predict `infrastructure_damage`. Tree 3 splits on "people" → "trapped" → predicts `missing_and_found_people`. Out of 200 trees, the majority votes the answer.

**Where it shows up in the wild.** Credit scoring, fraud detection, customer segmentation, ecology (species classification), wherever feature interactions matter and you don't want to engineer them by hand.

**Why it's in our comparison.** The **non-linear contrast**. Linear models can't natively model interactions like *"'rescue' AND 'helicopter' together strongly imply X, but neither word alone does"*. The forest can. If word interactions matter for tweet classification, the forest will out-score the linear models. If they don't, the forest will roughly tie or lose — an informative result either way.

In [6]:
rf_model, rf_time = train_model('Random Forest', MODEL_CONFIGS['Random Forest'], X_train, y_train)

[15:51:25]   training Random Forest …


[15:52:04]     Random Forest done in 39.0s


## Evaluation metrics — what we'll measure and why

We compute several metrics rather than one. Each tells a different story about the model. Concrete numbers below use the rough class proportions from our test split (~37% `not_humanitarian`, ~31% `other_relevant_information`, …, ~0.4% `missing_and_found_people`) so the trade-offs feel real.

### Accuracy
**What it measures.** Correct predictions ÷ total predictions. Plain head-count of how many test tweets the model labelled correctly.
**Worked example.** A model that predicts `not_humanitarian` for every single test tweet would score ≈ 0.37 (about 37% of test tweets really are `not_humanitarian`). It learned nothing.
**Where it falls short.** Massively misleading on imbalanced data. The trivial majority-class predictor above looks "40% good" while being completely useless.
**Verdict for our case.** **Reported only.** Never used to pick the winning model.

### Precision (per class)
**What it measures.** Of all tweets the model PREDICTED as class X, what fraction *actually were* class X. Formula: `TP / (TP + FP)`.
**Worked example.** If the model labels 100 tweets as `requests_or_needs` and 80 of them really were requests, precision = 0.80.
**Where it falls short.** A model that almost never predicts class X will trivially score high precision on it (e.g. flag only 1 tweet, get it right → precision=1.0) while missing 499 real ones. Precision says nothing about what you missed.
**Verdict.** **Diagnostic, per class.** Useful when reading the per-class report alongside recall.

### Recall (per class)
**What it measures.** Of all tweets that ACTUALLY ARE class X, what fraction did the model find. Formula: `TP / (TP + FN)`.
**Worked example.** If there are 200 real `requests_or_needs` tweets in the test set and the model catches 130, recall = 0.65 — it missed 35% of real requests.
**Where it falls short.** A model that flags everything as class X gets 100% recall on X but precision close to 0 — totally useless. Recall alone can be gamed by over-predicting.
**Verdict.** **Tiebreaker for the critical classes.** In disaster response, missing a real emergency costs lives; a false alarm costs five minutes — so we care more about recall than precision *for the critical classes specifically*.

### F1-score (per class)
**What it measures.** Harmonic mean of precision and recall. Formula: `F1 = 2 × P × R / (P + R)`. Punishes models that are good at one but bad at the other.
**Worked example.** P=0.80, R=0.60 → F1 = 2 × (0.80 × 0.60) / (0.80 + 0.60) = 0.96/1.40 ≈ 0.686. Compare with arithmetic mean (0.70): the harmonic mean is slightly lower, and the gap grows when P and R are far apart — exactly the behaviour we want.
**Where it falls short.** Single number per class hides which side of the trade-off is failing. Use alongside the per-class precision and recall, not as a replacement.
**Verdict.** **Per-class F1 is the headline diagnostic.** Aggregated forms (below) are how we compare whole models.

### Weighted F1
**What it measures.** F1 computed per class, then averaged with each class weighted by its support (how many true tweets it has). So big classes pull the average up or down more than small classes.
**Worked example.** `not_humanitarian` (37% of test data) F1=0.85, `missing_and_found` (0.4%) F1=0.10. Weighted F1 ≈ 0.85 × 0.37 + 0.10 × 0.004 + … — the small class barely shifts the number.
**Where it falls short.** Hides catastrophic failure on rare classes. A model can score weighted F1 = 0.72 while completely failing on `terrorism_related`.
**Verdict.** **Primary metric for picking the winning model** — the best single-number summary of overall performance.

### Macro F1
**What it measures.** Simple unweighted average of F1 across all classes. Treats every class equally, regardless of size.
**Worked example.** With 11 classes whose F1s are [0.85, 0.78, 0.72, 0.55, 0.50, 0.42, 0.38, 0.30, 0.20, 0.10, 0.05], macro F1 = 0.44. Weighted F1 on the same set might be ~0.72. The gap between them tells you how badly the model is doing on the rare classes.
**Where it falls short.** Penalises a model that aces the common classes but stumbles on a near-empty rare class. That penalty is *deserved* in our context, but you wouldn't use macro F1 alone if the rare class genuinely doesn't matter (it does for us).
**Verdict.** **Secondary metric.** Together with weighted F1, the gap reveals model behaviour on rare critical classes.

### Confusion matrix
**What it measures.** A square grid: rows = true class, columns = predicted class, cell = count (or proportion) of test tweets in that bucket. Diagonal cells are correct; off-diagonal cells are mistakes.
**Worked example.** A bright off-diagonal cell at (true=`requests_or_needs`, pred=`donation_and_volunteering`) tells you those two are visually being confused — exactly the pair our cosine-similarity EDA predicted would be tricky.
**Verdict.** **Diagnostic.** It's how you debug *what* a model is getting wrong, even if the F1 numbers look fine.

In [7]:
# Score every model and capture metrics + plots in one loop.
fitted = {
    'Naive Bayes':         {'model': nb_model,  'fit_seconds': nb_time},
    'Logistic Regression': {'model': lr_model,  'fit_seconds': lr_time},
    'Linear SVM':          {'model': svm_model, 'fit_seconds': svm_time},
    'Random Forest':       {'model': rf_model,  'fit_seconds': rf_time},
}
results = {}
for name, info in fitted.items():
    out = evaluate_model(info['model'], X_test, y_test, name)
    plot_confusion_matrix(y_test, out['y_pred'], labels, name)
    plot_classification_report(y_test, out['y_pred'], labels, name)
    results[name] = out

[15:52:04]   evaluating Naive Bayes …


[15:52:05]     saved /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/figures/confusion_naive_bayes.png
[15:52:05]     saved /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/figures/clsreport_naive_bayes.png
[15:52:05]   evaluating Logistic Regression …


[15:52:05]     saved /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/figures/confusion_logistic_regression.png


[15:52:06]     saved /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/figures/clsreport_logistic_regression.png
[15:52:06]   evaluating Linear SVM …


[15:52:06]     saved /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/figures/confusion_linear_svm.png


[15:52:06]     saved /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/figures/clsreport_linear_svm.png
[15:52:06]   evaluating Random Forest …


[15:52:09]     saved /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/figures/confusion_random_forest.png


[15:52:09]     saved /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/figures/clsreport_random_forest.png


In [8]:
# Side-by-side summary so we can scan the comparison fast.
summary = pd.DataFrame([{
    'model': name,
    'accuracy': r['accuracy'],
    'weighted_f1': r['weighted_f1'],
    'macro_f1': r['macro_f1'],
    'fit_seconds': fitted[name]['fit_seconds'],
} for name, r in results.items()]).set_index('model').round(4)
summary

,accuracy,weighted_f1,macro_f1,fit_seconds
model,,,,
Naive Bayes,0.6890,0.6613,0.3466,0.1554
Logistic Regression,0.6413,0.6633,0.4991,5.2938
Linear SVM,0.6985,0.7113,0.5133,5.7271
Random Forest,0.7307,0.7140,0.4774,39.0039


In [9]:
compare_models(results, save_path=os.path.join(FIGURES_DIR, 'model_comparison.png'))

[15:52:09]     saved /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/figures/model_comparison.png


'/Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/figures/model_comparison.png'

In [10]:
# Print full per-class classification report for each model — useful to
# see exactly which categories each model fails on.
for name, r in results.items():
    print(f'\n========== {name} ==========')
    print(r['report'])


========== Naive Bayes ==========
                                     precision    recall  f1-score   support

                affected_individual      0.677     0.397     0.500       693
                 caution_and_advice      0.706     0.197     0.308       583
                    disease_related      0.794     0.712     0.751       292
          displaced_and_evacuations      0.000     0.000     0.000        99
          donation_and_volunteering      0.639     0.511     0.568      1453
infrastructure_and_utilities_damage      0.711     0.181     0.289      1004
             injured_or_dead_people      0.730     0.396     0.513       561
           missing_and_found_people      0.000     0.000     0.000       103
                   not_humanitarian      0.739     0.871     0.799     10254
         other_relevant_information      0.612     0.749     0.674      8357
                    personal_update      0.000     0.000     0.000       204
                 physical_landslide     

## Picking the winner

### Metric hierarchy for disaster-tweet classification

1. **Weighted F1** — primary. Best single-number summary of overall classification quality on imbalanced data, and the metric we'll use to declare a winner.
2. **Macro F1** — secondary. Catches the failure mode where a model wins on weighted F1 by acing the two huge classes while ignoring the rare critical ones.
3. **Per-class recall on critical classes** (`injured_or_dead_people`, `missing_and_found_people`, `requests_or_needs`, `displaced_and_evacuations`, `terrorism_related`) — tiebreaker. In disaster response, missing a real emergency costs lives; a false alarm wastes five minutes. So we want recall on the categories where the cost of a miss is highest.
4. **Accuracy** — reported only. Not used for selection because of the imbalance: a model could score 39% by predicting `not_humanitarian` for everything.
5. **Why not recall alone?** A model that flagged every tweet as `injured_or_dead_people` would get 100% recall on it but precision near zero — useless. Recall is a tiebreaker, not a primary objective.
6. **Why we report training time.** Not a quality metric, but informs deployment. A 10× slower model that wins by 0.005 weighted F1 is rarely worth it in production.

In [11]:
# Final selection: highest weighted F1 wins.
best_name = max(results, key=lambda n: results[n]['weighted_f1'])
best = results[best_name]
best_model = fitted[best_name]['model']
print(f'Winner by weighted F1: {best_name}')
print(f'  weighted F1 = {best["weighted_f1"]:.4f}')
print(f'  macro F1    = {best["macro_f1"]:.4f}')
print(f'  accuracy    = {best["accuracy"]:.4f}')

# Honest secondary look: who wins on macro F1?
macro_winner = max(results, key=lambda n: results[n]['macro_f1'])
print(f'\nMacro-F1 winner (rare-class champion): {macro_winner}'
      f' (macro F1 = {results[macro_winner]["macro_f1"]:.4f})')

Winner by weighted F1: Random Forest
  weighted F1 = 0.7140
  macro F1    = 0.4774
  accuracy    = 0.7307

Macro-F1 winner (rare-class champion): Linear SVM (macro F1 = 0.5133)


### Reading the result

Random Forest takes the weighted-F1 crown but Linear SVM tends to lead on macro F1 — the classic split between *"who wins overall"* and *"who treats every class fairly"*. The forest's strength comes from being non-linear and ensemble-stable on the dominant classes; the SVM's strength comes from the explicit margin maximisation paired with `class_weight='balanced'` weighting, which keeps the rare classes alive.

**For deployment we'd actually consider both:**
- A *triage* deployment that sends critical tweets to human responders cares disproportionately about per-class recall on `injured_or_dead_people` / `missing_and_found_people` — Linear SVM wins there.
- A *bulk-tagging* deployment that just labels archives for analysis cares about average correctness — Random Forest wins there.

Per the metric hierarchy declared above, we lock in Random Forest as the official winner (weighted F1) and save it as `models/best_model.joblib` for the CLI.

In [12]:
# Per-class F1 for the winner — where does even the best model still fail?
plot_per_class_f1(y_test, best['y_pred'], labels, best_name)

[15:52:09]     saved /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/figures/per_class_f1_random_forest.png


'/Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/figures/per_class_f1_random_forest.png'

In [13]:
# Error analysis: ten random misclassifications from the winning model.
errors = get_misclassified(test_df, best['y_pred'], n=10)
for _, row in errors.iterrows():
    print(f'\nTRUE : {row["class_label"]}')
    print(f'PRED : {row["predicted"]}')
    print(f'TEXT : {row["text"][:200]}')


TRUE : infrastructure_and_utilities_damage
PRED : not_humanitarian
TEXT : LOOK: Ruby's winds down power lines in Albay  Strong winds from Typhoon Ruby (Hagupit) over ... http://t.co/LzFpMMJoIU #news #filipino

TRUE : requests_or_needs
PRED : not_humanitarian
TEXT : As conventional breeding has been unsuccessful in developing insect resistance in the cowpea, and smallholder farmers have limited access to costly insecticides, Feed the Future is working with partne

TRUE : displaced_and_evacuations
PRED : other_relevant_information
TEXT : RT @atomaraullo: Residents start fleeing to the hills in Quinapondan, Eastern Samar. #Hagupit #RubyPH @ANCALERTS http://t.co/kUk2FJrL4r

TRUE : requests_or_needs
PRED : not_humanitarian
TEXT : Please don't waste time coming forward, the situation is becoming complicated.

TRUE : requests_or_needs
PRED : other_relevant_information
TEXT : Thank you, Hurricane Sandy for giving me time to write to family and friends tomorrow. #handwrittenletters #SPcampusPr

In [14]:
# Save the winner so the CLI / downstream tooling can load it without
# re-running training.
save_model(best_model, vec, best_name)
save_model(best_model, vec, 'best_model')
print('Saved.')

[15:52:12]   saved Random Forest -> /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/models/random_forest.joblib


[15:52:14]   saved best_model -> /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/models/best_model.joblib
Saved.


## Honest weaknesses of the winning model

Even the best of these classical models has clear limits we should be transparent about:

- **Vocabulary brittleness.** TF-IDF only knows the words it saw in training. A new disaster using fresh slang, place names, or hashtags can produce tweets where most informative tokens are out-of-vocabulary, and the model has to guess.
- **No semantic awareness.** "Help is on the way" and "need help now" share the word *help* but have opposite operational implications. Bag-of-words representations cannot tell them apart; this is exactly where contextual models like BERT would help.
- **Label noise.** CrisisBench was assembled from several upstream datasets with different annotation rules. Some test errors are arguably ambiguous in the data itself.
- **English only.** A real Nepal-earthquake deployment would see substantial Nepali- and Hindi-language traffic that our pipeline drops on the floor.
- **No temporal awareness.** Disaster vocabulary evolves over the lifetime of an event (early casualty reports → later donation drives). A model trained statically on the full dataset has no notion of where in the disaster timeline a tweet sits.